# Future research proposal: Probability based hybrid nearest neighbors with full training set
My implementation of the probabilistic nearest neighbor classifier seems to not run for hours just to end up eating all 64GB of RAM like the original R version.
Here are some tests

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from pandas import read_csv
import numpy as np
import pandas as pd
from hybrids import PNeighborsClassifier

In [2]:
blackbox = 'rf'  # Random forest
# blackbox = 'ab'  # AdaBoosted decision trees
# blackbox = 'xg'  # XGboost (gbtree)

path = './datasets/census/'; name = 'census'
# path = './datasets/coupon/'; name = 'coupon'
# path = './datasets/stop&frisk/'; name = 'stop-and-frisk'

In [3]:
normalize = True

# read data
Xtrain = read_csv(path + 'train.csv',header = 0, sep = ',')
Xtest = read_csv(path + 'test.csv',header = 0, sep = ',')
bbm = read_csv(path + blackbox + '_ybtrain.csv',header = 0,sep = ',')
Ybtrain = np.array(bbm['Yb'])
bbm = read_csv(path + blackbox + '_ybtest.csv',header = 0,sep = ',')
Ybtest = np.array(bbm['Yb'])

ntrain = Xtrain.shape[0]
df = pd.concat([Xtrain, Xtest])
# Seperate Y (answers) from dataframe
Y = np.array(df['Y'])
df.drop(['Y'], axis = 1, inplace = True) 
numericals = [col for col in Xtrain.columns if Xtrain[col].dtype in [int, float] and (~Xtrain[col].isin([0, 1])).any()]

if normalize:
    # Normalize data
    df = df.astype({col: float for col in numericals})
    mean = Xtrain.loc[:, numericals].mean(axis=0)
    std = Xtrain.loc[:, numericals].std(axis=0) + 1e-8
    df.loc[:, numericals] = (df.loc[:, numericals] - mean) / std


ntrain = Xtrain.shape[0]
Ytrain = Y[:ntrain]
Ytest = Y[ntrain:]
Xtrain = df[:ntrain]
Xtest = df[ntrain:]


### Fitting my probabilistic nearest neighbor

In [4]:
model = PNeighborsClassifier(n_neighbors=21, gpu=True, n_jobs=-1, predict_chunksize=2000)
model.fit(Xtrain[:], Ytrain[:])

,n_neighbors,21
,predict_chunksize,2000
,gpu,True
,n_jobs,-1
,radius,1.0
,beta_max,10.0
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'


#### Probabilistic nearest neighbors' mean accuracy

In [5]:
result = model.predict(Xtest)
np.mean(result == Ytest)

np.float64(0.8483508384005897)

#### Normal k-nearest neighbors' mean accuracy

In [6]:
smodel = KNeighborsClassifier(n_neighbors=21)
smodel.fit(Xtrain, Ytrain)
sres = smodel.predict(Xtest)
np.mean(sres == Ytest)

np.float64(0.843928505620048)

As you can see the probabilistic nearest neighbors is achieved 0.5% better accuracy than the normal k-nearest neighbors, which is promising. I wrote in the thesis that the pnnclass R package crashes with full training set but this implementation makes experimenting with the probabilistic hybrid nearest neighbors with full training set feasible. It would still take marginally more time to perform all the feature and k selection with this probabilistic nearest neighbors, but someone with a supercomputer should fill the gap in my thesis.